# Camada Silver — `frequencia`

**Disciplina:** Mineração de Dados — FATEC Matão-SP (Prof. Rodrigo Vidrich)  
**Projeto:** Trabalho Final (P2) — Arquitetura Medallion (`Bronze → Silver → Gold → Power BI`)  
**Integrantes:** Jerônimo Barbieri Junior, Luan Chierato Oliveira, Miguel Miranda Morandini, Victor Hugo Malipense Testi  
**Data:** 13/11/2026  

**Técnicas aplicadas:** Padronização de Tipos (`id_aluno`, `data_aula`, `minutos_atraso`), Limpeza Textual (`curso`, `disciplina`, `turno`, `situacao`) e Registro de Escopo Amostral.

In [1]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

for candidate in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
    if (candidate / "notebooks" / "utils").exists():
        if str(candidate / "notebooks") not in sys.path:
            sys.path.insert(0, str(candidate / "notebooks"))
        break

from utils.silver_utils import (
    anonymize_name_lgpd,
    detect_outliers_iqr,
    detect_outliers_zscore,
    fix_mojibake_text,
    get_silver_paths,
    load_bronze,
    normalize_id_aluno,
    parse_br_float,
    parse_date_col,
    save_silver_outputs,
)

ROOT_DIR, BRONZE_DIR, SILVER_DIR = get_silver_paths()

In [2]:
df = load_bronze("frequencia", BRONZE_DIR)
rows_in = len(df)

df["id_frequencia"] = df["id_frequencia"].astype(str).str.strip().astype("string")
df["id_aluno"] = normalize_id_aluno(df["id_aluno"])
df["data_aula"] = parse_date_col(df["data_aula"])
df["minutos_atraso"] = pd.to_numeric(df["minutos_atraso"], errors="coerce").clip(lower=0).fillna(0).astype("Int64")

for col in ["curso", "disciplina", "turno", "situacao"]:
    df[col], _ = fix_mojibake_text(df[col])
    df[col] = df[col].str.title().astype("category")

dups_removed = int(df.duplicated(subset=["id_frequencia"]).sum())
df = df.drop_duplicates(subset=["id_frequencia"]).reset_index(drop=True)

log = {
    "dataset": "frequencia",
    "rows_in": rows_in,
    "rows_out": int(len(df)),
    "duplicates_removed": dups_removed,
    "escopo": "Micro-amostra complementar (20 registros / 9 alunos).",
}
save_silver_outputs(df, log, "frequencia", SILVER_DIR)
print(json.dumps(log, ensure_ascii=False, indent=2))
df.head(5)

{
  "dataset": "frequencia",
  "rows_in": 20,
  "rows_out": 20,
  "duplicates_removed": 0,
  "escopo": "Micro-amostra complementar (20 registros / 9 alunos)."
}


,id_frequencia,id_aluno,data_aula,curso,disciplina,turno,situacao,minutos_atraso
0,FREQ000000001,10245,2024-02-15,Engenharia De Software,Algoritmos E Programação,Matutino,Presente,0
1,FREQ000000002,83921,2024-02-15,Administração,Gestão Estratégica,Noturno,Atrasado,15
2,FREQ000000003,45210,2024-02-15,Direito,Direito Constitucional,Vespertino,Ausente,0
3,FREQ000000004,12093,2024-02-15,Medicina,Anatomia Humana,Matutino,Presente,0
4,FREQ000000005,73104,2024-02-15,Engenharia De Software,Bancos De Dados,Noturno,Atrasado,8
